# Anyways uhhh yeah fuckin' uhhh
## chords

So yeah, I like the code I did yesterday. But what it's currently generating isn't particularly interesting.

There are only certain scales I care about. Mostly just major, melodic minor, harmonic minor, etc.

### RANDOM FUCKING THOUGHT

I think um. uhhhh. fuckin'. uhhh... uhhhh. Oh yeah. So, walking a set of voices through a scale is cool.
That's how you can move through a scale diatonically.

But sometimes, there are interesting movements that aren't diatonic - some notes leave the scale, but there is a pattern.

How do we model this?

One idea is just to partition the voices into different groups, and each group gets a scale.

So like, the notes form a C major 7 chord. C E G B. Well, you could partition the chord into (C B) and (E G). 

(C B) moves along the C dim scale while (E G) moves along the D dim scale. Or something like that. That gives the Cmaj7, Bmin7, Amaj7, Abmin7, Gbmaj7, etc pattern. It's a pattern, despite not all the notes fitting into a single scale (well... besides the chromatic scale)

Okay. Cool. So that's a whole thing that needs to be modeled. Because it's like, how do you describe it? I guess you have a "seed" scale, and the pitch classes are partitioned into groups that are assigned different scales. The simplest form of this is the diatonic form, where all the pitch classes are in a single partition, and they all belong to the scale that's assigned to them.

Well, you could have the partitions themselves be pcsets. 

## WTF Am I Talking About?

Well, what objects am I describing here?

There's a scale - we all know about those. Represented by a pitch class set or a scale function, they consist of a repeating pattern of pitch classes.

But something like `[Cm, Bb, Am, G, F#m, E, Ebm, Db]` is weird to model. 

The same object can be represented in a bunch of different ways.

The list above could, itself, describe the whole pattern.

Or you could use a pitch function.

### Wait, pause... what about pitch class functions?

Oh. Uh... I mean, that sounds cool. 

So, a pitch function is something like `[2,4,5,7,9,11,12] + 2` (for D major). 
But it can also have elements out of order, and non-positive. 
So `[2,-1,5,3,-5] + 1` is a pitch function, too.

But would it be derivative of a pitch class function? 
Mod 12, that pattern would be `[2, 11, 5, 3, 7] + 1`.
Which you could also represent as `[3, 0, 6, 4, 8]`.

In fact, so far it seems that there's not much use to using the transposition notation (+) in pitch class functions, and that they seem almost entirely identical to pitch class cycles.

### Oh okay. Anyways

Yeah. So, anyways, these patterns can be represented in different ways.

`[Cm, Bb, Am, G, F#m, E, Ebm, Db]` For example. If you look at the pitch class sets...

```
 0  3  7
10  2  5
 9  0  4
 7 11  2
 6  9  1
 4  8 11
 3  6 10
 1  5  8
```

You can see that transposing it (like a matrix, not like a pitch object) yields 3 different pitch class cycles glued together. What do you call this? A polycycle is a fun way to describe it. But it's the same as a pitch class set cycle. But yeah... polycycle is what I'll call this for now.

You could have a set of cycles that aren't assumed to be glued together at a certain position; you just specify the indices of the cycles that you want them all glued together at. 

# Back to the progression machine

Something I think would be more useful is if the algorithm we built yesterday handled scale changes differently. 

I do like the idea of scale changes being decided on the fly along with everything else. As opposed to deciding the scale changes, then deciding the chords pulled from the scale, etc. But either approach works. I do want to be able to feed in an existing scale progression to the chord progression builder.

The question is, what scales do you choose? But I think it's good to look at what changing scales _does_ to the notes you can choose.

## Let's simplify 

I think I'm complicating things too much. Let's get some easy chords. Let's only use major keys.

I need a function that takes a pitch class set and returns the set of all its transpositions.

In [ ]:
from harmonica.pitch._scales import PitchClassSet


c_maj_scale = PitchClassSet([0, 2, 4, 5, 7, 9, 11], 12)
transpositions = c_maj_scale.get_transpositions()
for transposition in transpositions:
    print(transposition)

There we go! So now we need to collect a pool of different types of scales we like, union their transpositions together, and we'll have a set of scales to draw from.

In [ ]:
interesting_scales = [
    PitchClassSet(pcs, 12)
    for pcs in [[0, 2, 4, 5, 7, 9, 11], [0, 2, 3, 5, 7, 9, 11], [0, 2, 3, 5, 7, 8, 11]]
]


def get_transpositions(pcsets: list[PitchClassSet]) -> list[PitchClassSet]:
    """Takes a list of pitch class sets, gets all the transpositions of each one,
    and returns a combined list of all of them."""

    combined_transpositions = []

    for pcset in pcsets:
        pcset_transpositions = pcset.get_transpositions()
        for transposition in pcset_transpositions:
            if transposition not in combined_transpositions:
                combined_transpositions.append(transposition)

    return combined_transpositions


for pcset in get_transpositions(interesting_scales):
    print(pcset)

Great.

This will provide the set of all scales of interest.

Now, we need to figure out how we're deriving scale changes.

Here's one way of thinking about things. You have scale changes, and within those changes, you have chord changes. Sometimes, you have a scale change too. Hell, sometimes you have a scale change without a chord change, if you consider that there is other harmonic "machinery" outside of a particular chord changing operation.

So, how do we come up with scale changes? How can you change scales?

To keep things simple, let's just focus on the major scale.

You can model scale changes, in this case, by just choosing a transposition.

Let's not even have scale changes, to start. Let's just have chord changes, and a single scale.

### On making Interesting Choices

Making music is all about making interesting choices.

Some choices are more interesting, some features are more salient.

I can't reasonably expect a computer to imitate these decisions (yet), 
but I can build algorithms that depend on data I feed it, data that tells 
the algorithm what to look for. 

I can do one of two things: I can just generate random chord changes, or I can 
tell it chord changes that I like.

Chord changes don't mean much without a harmonic rhythm.

Well, what chord changes do I even like?

Am Em F C Dm Am G

Maybe this is a question to investigate AFTER implementing the random approach first. The random approach won't be that interesting, but it will at least create a starting point.

So, how do I randomly select chords for the progression?

One way is to just have a steady 1-bar harmonic rhythm.

Decide on some number of bars, like 8. Then just generate random chords within a major scale.

In [2]:
import random

from harmonica.pitch._changes import PCSetSeq
from harmonica.pitch import PitchClassSet
from harmonica.pitch._melody import PCSequence


def ez_progression_builder(scale: PitchClassSet, pcseq: PCSequence) -> PCSetSeq:
    """Builds a chord progression (in the form of a sequence of pitch class sets)
    where the chords all belong to a particular scale, and the progression
    embeds a sequence of pitch classes (which can represent something like a bassline.)
    """

    progression: list[PitchClassSet] = []

    # For each pitch class in the sequence, choose 2 other pitch classes
    # from the scale at random.

    for pitch_class in pcseq.pitch_classes:
        # These are the other pitch classes to choose from
        other_pcs: list[int] = [pc for pc in scale.pitch_classes if pc != pitch_class]
        # Sample two of them at random
        sample = random.sample(other_pcs, 2)
        # Combine with pitch class from sequence to yield the chord
        chord_pcs = sorted([pitch_class] + sample)
        next_chord = PitchClassSet(chord_pcs, 12, pitch_class)
        progression.append(next_chord)

    return PCSetSeq(progression)


cmaj = PitchClassSet([0, 2, 4, 5, 7, 9, 11], 12, 0)
pcseq = PCSequence([5, 4, 0, 2], 12)
progression = ez_progression_builder(cmaj, pcseq)

for chord in progression:
    print(chord)

PitchClassSet(pitch_classes=[0, 2, 5], modulus=12, root=5)
PitchClassSet(pitch_classes=[0, 4, 9], modulus=12, root=4)
PitchClassSet(pitch_classes=[0, 9, 11], modulus=12, root=0)
PitchClassSet(pitch_classes=[2, 4, 11], modulus=12, root=2)


Okay, cool. Now we have to turn the pitch class sets into pitch sets.
This requires voicing the pitch class sets.

## Pitch class set voicing

Take a C major triad. What are all the ways you can voice this?

I've covered this before [here.](../../ideas/chord_voicings.ipynb)

All you really have to do is take permutations of the pcset.

So a triad can be voiced in 3! = 6 ways.

So, say the pcset is [2,5,9]. 

1. Determine how many times each pitch class will occur in the voicing. Example: 2 occurs 1 time, 5 occurs 2 times, and 9 occurs 2 times.
2. Permute the multiset. Example: 9 5 2 9 5.
3. Insert octaves between notes. Octaves are represented by the modulus. Example: 9 5 2 12 9 5.
4. Determine octave of bass pitch. Example: Second octave, 33.
5. Build pitch set, one pair of consecutive pitch classes at a time. The interval of 9 to 5 is given by (5 - 9) mod 12 = 8. So we add 8 to 33 and get the next pitch in the set, 41. (Note: If a number appears twice in a row in the permutation, it represents two notes an octave apart.)
6. Continue this to get the pitch set: 33 41 50 69 77.

I reckon this procedure can derive every pitch set that belongs to a pitch class set. 

The question is now, how do I select things like, how many times each pitch class appears, how the pitch classes are permuted, how many octaves are inserted, which octave is the chord in...

In practice, I think the seeded generation approach is a close approximation to how I select things. I'm usually looking at the previous chord and thinking about the movement of the voices. I usually look at the voices of the previous chord, and look for pitches in the neighborhood of that voice's pitch.

So if the previous voicing was [0,3,7], and the next pitch class set is [2,5,11], then the closest pitch set to [0,3,7] would be [-1,2,5]. I would determine that by going through each pitch in [0,3,7] and asking, what's the 1, 2, 3 etc semitone neighborhood around 0 in the pcset [2,5,11]. This question is one that is useful to ask sometimes. Getting pitch sets in the neighborhood of another pitch set that belong to a given pitch class set.